# DIP Pipeline — Vehicle Verification System
### Digital Image Processing notebook demonstrating all concepts used
Covers: U1 (Interpolation, Gray-level, Histogram, Spatial Filtering), U2 (DFT, DCT, Wavelets), U3 (Segmentation)


In [ ]:
import sys
sys.path.insert(0, '../backend')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dip.preprocessor import full_dip_preprocess
from dip.frequency import apply_dft_lowpass, apply_dct_enhance, apply_wavelet_denoise, get_dft_spectrum
from dip.segmentation import detect_plate_region, preprocess_plate_for_ocr

# Load a sample image — replace with your dataset image
# IMAGE_PATH = '../dataset/your_image.jpg'
IMAGE_PATH = 'sample_plate.jpg'  # Replace with actual path

print('Imports OK!')

## U1-T1: Interpolation and 2D Signals
Bicubic interpolation for standardizing image resolution.

In [ ]:
img = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Bicubic (high quality) vs Nearest (blocky)
nearest = cv2.resize(img_rgb, (320, 240), interpolation=cv2.INTER_NEAREST)
bicubic = cv2.resize(img_rgb, (320, 240), interpolation=cv2.INTER_CUBIC)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb); axes[0].set_title('Original')
axes[1].imshow(nearest); axes[1].set_title('Nearest Neighbor')
axes[2].imshow(bicubic); axes[2].set_title('Bicubic (Used)')
plt.tight_layout(); plt.show()

## U1-T2: Gray-Level Transformations
Grayscale conversion + Gamma correction for lighting normalization.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Gamma correction
def gamma_correct(img, gamma=1.5):
    table = np.array([(i/255.0)**(1/gamma)*255 for i in range(256)]).astype('uint8')
    return cv2.LUT(img, table)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img_rgb); axes[0].set_title('Original')
axes[1].imshow(gray, cmap='gray'); axes[1].set_title('Grayscale')
axes[2].imshow(gamma_correct(gray, 0.5), cmap='gray'); axes[2].set_title('Gamma 0.5 (Darken)')
axes[3].imshow(gamma_correct(gray, 2.0), cmap='gray'); axes[3].set_title('Gamma 2.0 (Lighten)')
plt.tight_layout(); plt.show()

## U1-T3: Histogram Analysis and CLAHE
Plotting histograms and comparing global vs adaptive histogram equalization.

In [ ]:
hist_orig = cv2.calcHist([gray], [0], None, [256], [0, 256])
equalized = cv2.equalizeHist(gray)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_img = clahe.apply(gray)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0][0].imshow(gray, cmap='gray'); axes[0][0].set_title('Original Gray')
axes[0][1].imshow(equalized, cmap='gray'); axes[0][1].set_title('Global HE')
axes[0][2].imshow(clahe_img, cmap='gray'); axes[0][2].set_title('CLAHE (Used)')

for ax, img_hist, title in zip(axes[1], [gray, equalized, clahe_img], ['Original', 'Global HE', 'CLAHE']):
    hist = cv2.calcHist([img_hist], [0], None, [256], [0, 256])
    ax.plot(hist, color='cyan'); ax.set_title(f'Histogram - {title}')
    ax.set_xlim([0, 256])

plt.tight_layout(); plt.show()

## U2-T2 & U2-T4: DFT and Frequency Domain Filtering

In [ ]:
from dip.frequency import apply_dft_lowpass, apply_dft_highpass, get_dft_spectrum

spectrum = get_dft_spectrum(gray)
lp = apply_dft_lowpass(gray, cutoff=30)
hp = apply_dft_highpass(gray, cutoff=30)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(spectrum, cmap='hot'); axes[1].set_title('DFT Spectrum')
axes[2].imshow(lp, cmap='gray'); axes[2].set_title('Low-Pass (Smoothed)')
axes[3].imshow(hp, cmap='gray'); axes[3].set_title('High-Pass (Edges)')
plt.tight_layout(); plt.show()

## U2-T4: Wavelet Denoising (Filter Banks)

In [ ]:
import pywt
from dip.frequency import apply_wavelet_denoise

# Show wavelet decomposition
coeffs = pywt.dwt2(gray.astype(np.float64), 'haar')
cA, (cH, cV, cD) = coeffs

denoised = apply_wavelet_denoise(gray)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0][0].imshow(gray, cmap='gray'); axes[0][0].set_title('Original')
axes[0][1].imshow(np.abs(cH), cmap='gray'); axes[0][1].set_title('Horizontal Detail (cH)')
axes[0][2].imshow(np.abs(cV), cmap='gray'); axes[0][2].set_title('Vertical Detail (cV)')
axes[1][0].imshow(np.abs(cD), cmap='gray'); axes[1][0].set_title('Diagonal Detail (cD)')
axes[1][1].imshow(cA, cmap='gray'); axes[1][1].set_title('Approximation (cA)')
axes[1][2].imshow(denoised, cmap='gray'); axes[1][2].set_title('Wavelet Denoised')
plt.tight_layout(); plt.show()

## U3-T3: Image Segmentation — Plate Detection

In [ ]:
from dip.segmentation import detect_plate_region, preprocess_plate_for_ocr

plate_crop, bbox = detect_plate_region(img)
print(f'Detected plate bbox: {bbox}')

if plate_crop is not None:
    plate_ocr = preprocess_plate_for_ocr(plate_crop)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(img_rgb); axes[0].set_title('Full Vehicle Image')
    p = plate_crop if len(plate_crop.shape)==3 else cv2.cvtColor(plate_crop, cv2.COLOR_GRAY2RGB)
    axes[1].imshow(cv2.cvtColor(p, cv2.COLOR_BGR2RGB) if len(p.shape)==3 else p, cmap='gray')
    axes[1].set_title(f'Segmented Plate\n{bbox}')
    axes[2].imshow(plate_ocr, cmap='gray'); axes[2].set_title('Preprocessed for OCR')
    plt.tight_layout(); plt.show()

## Full DIP Pipeline

In [ ]:
result = full_dip_preprocess(IMAGE_PATH)

stages = ['original', 'grayscale', 'denoised', 'clahe', 'sharpened', 'binary']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, key in zip(axes.flat, stages):
    img_show = result[key]
    if len(img_show.shape) == 3:
        ax.imshow(cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB))
    else:
        ax.imshow(img_show, cmap='gray')
    ax.set_title(key.upper())
    ax.axis('off')

plt.suptitle('Full DIP Pipeline', fontsize=16)
plt.tight_layout()
plt.show()